# 007-search-prior-and-gate — analysis

# Imports

In [ ]:
import json
import os
from dataclasses import replace

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.benchmark import paired_frame_bootstrap
from uavloc.matching.correlator import Correction, PositionCorrelator
from uavloc.plotting.correlation import plot_heatmap, plot_uncertainty
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "007-search-prior-and-gate"
BASELINE = "007.1"
RULE_A_RUN = "007.4"
RULE_B_RUN = "007.7"
GATE_ALONE_RUN = "007.8"
DIAGNOSTIC_RUNS = ["007.10"]
LEVEL_RUNS = ["007.1", "007.2", "007.3", "007.4", "007.5", "007.6"]
REFERENCE_005_3 = ("005-village-crossdate", "005.3")
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
RULE_A_GAIN_M = 2.0
RULE_A_MAX_LOSS_M = 0.5
RULE_B_HARM_RATIO = 0.5
RULE_B_MIN_APPLIED = 0.5
RULE_B_MAX_COST_M = 0.5
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42
S_MAX_SWEEP_M = list(np.round(np.arange(0.5, 12.01, 0.25), 2))
GALLERY_RUN = "007.7"
GALLERY_NOISE_M = 10.0
GALLERY_MUST = ("birky-15", 15.0)
N_SIGMA = 2.0
DEVICE = "mps"

# Data

In [ ]:
runs = {r["run"]: r for r in load_runs(EXP) if r["status"] == "ok"}
records = {run: pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{run}.csv") for run in runs}
list(runs)

In [ ]:
def describe(run: dict) -> str:
    c = run["config"]
    window = f"rect x{c['search_scale']:g}" if c["window"] == "rect" else f"round R {c['search_radius_m']:g} m"
    prior = "flat" if c["prior_sigma_m"] is None else f"sigma {c['prior_sigma_m']:g} m"
    tuned = " (tau with gate)" if c["tuning"].get("temperature_by") == "gate" else ""
    return f"{window} | {prior} | gate {c['gate']}{tuned}"

In [ ]:
def metric(run: str, key: str, noise_m: float) -> float:
    return runs[run]["metrics"][f"{key}_at_{noise_m:g}m"]

# Run table

In [ ]:
table = pd.DataFrame({run: {"setup": describe(r),
                            **{f"@{n:g} m": metric(run, "mean_position_error_m", n) for n in NOISE_M},
                            "harm @10": metric(run, "harm_share", 10.0),
                            "applied @10": metric(run, "accept_share", 10.0) + metric(run, "across_share", 10.0),
                            "across @10": metric(run, "across_share", 10.0),
                            "reach @15": metric(run, "reachable_share", 15.0),
                            "ms/corr (cpu)": r["metrics"]["ms_per_correction"]}
                      for run, r in runs.items()}).T
table.round(3)

In [ ]:
reference = next(r for r in load_runs(REFERENCE_005_3[0]) if r["run"] == REFERENCE_005_3[1])
{"005.3 @15 m": reference["metrics"]["mean_position_error_m_at_15m"], "005.3 map_date": reference["config"].get("map_date"),
 "007.2 @15 m": metric("007.2", "mean_position_error_m", 15.0), "007.2 map_date": runs["007.2"]["config"]["map_date"]}

In [ ]:
per_place = pd.concat({run: r.groupby(["place", "noise_m"]).apply(lambda g: pd.Series({
    "mean_m": g["error_m"].mean(), "harm": (g["error_m"] > g["prior_error_m"] + 2).mean(),
    "applied": g["decision"].isin(["accept", "across"]).mean()})) for run, r in records.items()})
per_place.xs(PRIMARY_NOISE_M, level="noise_m").unstack("place").round(3)

# Tuned values

In [ ]:
pd.DataFrame({run: {f"{key} [{place.split('-')[0]}]": value
                    for key in ("temperature", "gate_s_max_m", "gate_c_min")
                    for place, value in r["config"][key].items()}
              for run, r in runs.items()}).T

# Paired bootstrap

In [ ]:
def vs(run: str, reference: str, noise_m: float) -> dict[str, float]:
    return paired_frame_bootstrap(records[run], records[reference], noise_m, N_BOOTSTRAP, CI_PCT, SEED)

In [ ]:
vs_baseline = pd.DataFrame({(run, n): vs(run, BASELINE, n) for run in runs if run != BASELINE for n in NOISE_M}).T
vs_baseline.round(2)

In [ ]:
vs_rule_a = pd.DataFrame({(run, n): vs(run, RULE_A_RUN, n) for run in runs if run != RULE_A_RUN for n in NOISE_M}).T
vs_rule_a.round(2)

# Rules

In [ ]:
rule_a = {n: vs(RULE_A_RUN, BASELINE, n) for n in NOISE_M}
rule_a_pass = (rule_a[15.0]["diff_m"] <= -RULE_A_GAIN_M and rule_a[15.0]["ci_hi"] < 0
               and all(rule_a[n]["ci_hi"] < RULE_A_MAX_LOSS_M for n in (5.0, 10.0)))
{**{f"{RULE_A_RUN} - {BASELINE} @{n:g}": (round(d["diff_m"], 2), round(d["ci_lo"], 2), round(d["ci_hi"], 2))
    for n, d in rule_a.items()}, "rule A passes": bool(rule_a_pass)}

In [ ]:
def rule_b(run: str) -> dict[str, float | bool]:
    harm, harm_ref = metric(run, "harm_share", 10.0), metric(RULE_A_RUN, "harm_share", 10.0)
    applied = metric(run, "accept_share", 10.0) + metric(run, "across_share", 10.0)
    cost = metric(run, "mean_position_error_m", 10.0) - metric(RULE_A_RUN, "mean_position_error_m", 10.0)
    checks = {"harm halved": harm <= RULE_B_HARM_RATIO * harm_ref, "applied >= 50%": applied >= RULE_B_MIN_APPLIED,
              "cost <= 0.5 m": cost <= RULE_B_MAX_COST_M}
    return {"harm": harm, f"harm {RULE_A_RUN}": harm_ref, "applied": applied, "cost_m": cost, **checks,
            "passes": all(checks.values())}

In [ ]:
pd.DataFrame({run: rule_b(run) for run in [RULE_B_RUN, *DIAGNOSTIC_RUNS] if run in runs}).T

# Plots

In [ ]:
fig_levels, ax = plt.subplots(figsize=(6.5, 4.2))
for run in LEVEL_RUNS:
    ax.plot(NOISE_M, [metric(run, "mean_position_error_m", n) for n in NOISE_M], "o-", label=f"{run}  {describe(runs[run])}")
ax.plot(NOISE_M, NOISE_M, "k:", lw=1, label="no correction")
ax.set_xlabel("injected error, m"); ax.set_ylabel("mean position error, m"); ax.set_xticks(NOISE_M)
ax.legend(fontsize=7); ax.grid(alpha=.3); fig_levels.tight_layout()
fig_levels.savefig(experiment_dir(EXP) / "figures" / "error-vs-level.png", dpi=110)

In [ ]:
def risk_coverage(run: str, noise_m: float) -> pd.DataFrame:
    """Accept-only sweep of s_max at each fold's c_min: share of draws accepted and their mean error."""
    r = records[run][(records[run]["noise_m"] == noise_m)]
    c_min = r["place"].map(runs[run]["config"]["gate_c_min"])
    eligible = (r["status"] == "ok") & (r["peak_z"] != 0) & (r["coverage"] >= c_min)
    rows = []
    for s_max in S_MAX_SWEEP_M:
        accepted = eligible & (r["s1_m"] <= s_max)
        rows.append({"s_max_m": s_max, "accepted_share": accepted.mean(),
                     "mean_error_accepted_m": r.loc[accepted, "map_error_m"].mean()})
    return pd.DataFrame(rows)

In [ ]:
fig_risk, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, noise_m in zip(axes, (10.0, 15.0)):
    for run in (RULE_B_RUN, GATE_ALONE_RUN):
        curve = risk_coverage(run, noise_m)
        ax.plot(curve["accepted_share"], curve["mean_error_accepted_m"], ".-", label=f"{run}  {describe(runs[run])}")
    ax.axhline(noise_m, color="k", ls=":", lw=1)
    ax.set_title(f"{noise_m:g} m injected, sweep s_max {S_MAX_SWEEP_M[0]}..{S_MAX_SWEEP_M[-1]} m", fontsize=9)
    ax.set_xlabel("share of draws accepted"); ax.grid(alpha=.3)
axes[0].set_ylabel("mean error of accepted draws, m"); axes[0].legend(fontsize=7)
fig_risk.tight_layout()
fig_risk.savefig(experiment_dir(EXP) / "figures" / "risk-coverage.png", dpi=110)

# Gallery

In [ ]:
config = runs[GALLERY_RUN]["config"]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}
maps = {place: load_date(place, config["map_date"]) for place in config["places"]}
model = load_segmenter(REPO_ROOT / config["seg_checkpoint_sat"], DEVICE, config["seg_architecture"])

In [ ]:
def fold_correlator(place: str) -> PositionCorrelator:
    """The gallery run's correlator with the values tuned for `place`'s fold."""
    tuned = {key: config[key][place] for key in ("temperature", "gate_s_max_m", "gate_c_min") if config[key][place] is not None}
    features = SegmentationFeatures(model, model, maps[place], config["seg_channels"], config["match_gsd_m"],
                                    config["match_gsd_m"], DEVICE)
    return PositionCorrelator(features, replace(correlator_config({**config, "gate": "none"}), gate=config["gate"], **tuned))

In [ ]:
correlators = {place: fold_correlator(place) for place in config["places"]}

In [ ]:
g = records[GALLERY_RUN]
at_level = g[(g["noise_m"] == GALLERY_NOISE_M) & (g["peak_z"] != 0)]
picks = pd.concat([pd.concat([d.nsmallest(1, "error_m"), d.nlargest(1, "error_m")]) for _, d in at_level.groupby("decision")])
must = g[(g["frame"] == GALLERY_MUST[0]) & (g["noise_m"] == GALLERY_MUST[1])].nlargest(1, "error_m")
picks = pd.concat([picks, must])
picks[["frame", "place", "noise_m", "bearing_deg", "decision", "error_m", "map_error_m", "s1_m", "s2_m", "coverage"]].round(2)

In [ ]:
def gallery_panel(ax: plt.Axes, row: pd.Series) -> float:
    """One draw's heatmap on its patch with the posterior ellipse; returns the recomputed error (m) as a check."""
    frame, prior = frames[row["frame"]], LatLon(row["prior_lat"], row["prior_lon"])
    correlator = correlators[row["place"]]
    correction = correlator.correct(frame, prior)
    patch = maps[row["place"]].crop_rotated(prior, frame.yaw_deg, *correction.patch_size_m, out_mpp=config["match_gsd_m"])
    true_offset = correlator.patch_offset_m(prior, frame.position, frame.yaw_deg)
    s1, s2 = correction.uncertainty_m
    plot_heatmap(ax, patch, correction, true_offset, "red",
                 f"{row['frame']} @{row['noise_m']:g} m: {correction.decision}\n"
                 f"s1 {s1:.1f} / s2 {s2:.1f} m, err {row['error_m']:.1f} m (prior {row['prior_error_m']:.0f})")
    plot_uncertainty(ax, correction, "white", N_SIGMA, config["search_radius_m"])
    return haversine_m(correction.position, frame.position)

In [ ]:
fig_gallery, axes = plt.subplots(2, 4, figsize=(17, 8.5))
recomputed = [gallery_panel(ax, row) for ax, (_, row) in zip(axes.flat, picks.iterrows())]
for ax in axes.flat[len(picks):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7, loc="lower left")
fig_gallery.tight_layout()
fig_gallery.savefig(experiment_dir(EXP) / "figures" / "gallery.png", dpi=100)
{"max |recomputed - logged| error, m": float(np.max(np.abs(np.array(recomputed) - picks["error_m"].to_numpy())))}